In [3]:
# =============== TEXT-ONLY (No-FAISS): e5 embeddings + sklearn kNN + TF-IDF Ridge ===============
# • Fast, robust baseline with OOF SMAPE and auto-blended test preds
# • GPU (if available) ONLY speeds up embedding; kNN/TF-IDF/Ridge run on CPU cores
# • Checkpoints: caches embeddings and writes OOF/intermediate artifacts; safe to resume
# -----------------------------------------------------------------------------------------------
from pathlib import Path
CSV_DIR = Path("/content/drive/MyDrive/ml/ML challenge data/dataset")   # <-- edit if needed
OUT_DIR = Path("/content/drive/MyDrive/ml/text_best_cpu_v1")           # change per account when parallelizing
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Minimal deps (uncomment in Colab) ---
!pip -q install -U sentence-transformers scikit-learn scipy joblib

import os, gc, json, random, numpy as np, pandas as pd, joblib
from tqdm.auto import tqdm
import torch
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import NearestNeighbors
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from scipy.sparse import hstack

# ---------------- Device & threads ----------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE=="cuda" else "")
if DEVICE == "cpu":
    CORES = max(4, os.cpu_count() or 4)
    os.environ.update({
        "OMP_NUM_THREADS": str(CORES), "MKL_NUM_THREADS": str(CORES),
        "OPENBLAS_NUM_THREADS": str(CORES), "NUMEXPR_NUM_THREADS": str(CORES),
        "VECLIB_MAXIMUM_THREADS": str(CORES),
    })
    torch.set_num_threads(CORES); torch.set_num_interop_threads(max(1, CORES//2))
else:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

# ---------------- Config ----------------
SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
N_FOLDS=5
MODEL_NAME = "intfloat/e5-base-v2"       # strong, plug-and-play
TEXT_PREFIX = "passage: "                # e5 works best with a prefix
BATCH_ENC = 1024 if DEVICE=="cuda" else 256
K_NEIGHBORS=32
ALPHA=2.0                                # weight neighbors by (cos_sim)^ALPHA
WORD_MAX=200_000; CHAR_MAX=100_000
RIDGE_ALPHA=3.0

CACHE = OUT_DIR/"cache"; CACHE.mkdir(parents=True, exist_ok=True)
MODEL_DIR = OUT_DIR/"model"; MODEL_DIR.mkdir(parents=True, exist_ok=True)

# ---------------- Data ----------------
train = pd.read_csv(CSV_DIR/"train.csv")
test  = pd.read_csv(CSV_DIR/"test.csv")

CAND = ["catalog_content","description","product_description","title","name","product_name"]
def pick_text_col(df):
    low={c.lower():c for c in df.columns}
    for c in CAND:
        if c in low: return low[c]
    raise RuntimeError(f"No text column found; expected one of {CAND}")
TEXT_COL = pick_text_col(train)
assert TEXT_COL in test.columns, f"Test missing {TEXT_COL}"
train["__text__"] = train[TEXT_COL].fillna("").astype(str)
test["__text__"]  = test[TEXT_COL].fillna("").astype(str)
mask_tr = train["__text__"].str.strip()!=""

y_full = train["price"].values.astype(float)

# ---------------- Embeddings (cached) ----------------
TRN_EMB = CACHE/"train_emb.npy"; TST_EMB = CACHE/"test_emb.npy"; TRN_IDX = CACHE/"train_idx.npy"
if TRN_EMB.exists() and TST_EMB.exists() and TRN_IDX.exists():
    Xtr = np.load(TRN_EMB, mmap_mode="r"); Xte = np.load(TST_EMB, mmap_mode="r")
    tr_idx = np.load(TRN_IDX); print("Loaded cached embeddings:", Xtr.shape, Xte.shape)
else:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer(MODEL_NAME, device=DEVICE); model.max_seq_length = 256

    # Prefix EACH string, not the whole list
    tr_series = train.loc[mask_tr, "__text__"].fillna("").astype(str)
    te_series = test["__text__"].fillna("").astype(str)
    tr_texts = [f"{TEXT_PREFIX}{t}" for t in tr_series.tolist()]
    te_texts = [f"{TEXT_PREFIX}{t}" for t in te_series.tolist()]

    def encode(texts, path):
        embs = model.encode(
            texts,
            batch_size=BATCH_ENC,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=True
        ).astype("float32")
        np.save(path, embs)
        return embs

    print("Encoding TRAIN..."); Xtr = encode(tr_texts, TRN_EMB)
    print("Encoding TEST ...");  Xte  = encode(te_texts, TST_EMB)
    tr_idx = train.index[mask_tr].to_numpy(dtype=int); np.save(TRN_IDX, tr_idx)
    del model; gc.collect()

y_sub = y_full[tr_idx]; ylog_sub = np.log1p(y_sub)

# ---------------- kNN (cosine) helpers ----------------
def knn_predict_emb(X_train, ylog_train, X_query, k=K_NEIGHBORS, alpha=ALPHA):
    nn = NearestNeighbors(n_neighbors=k, metric="cosine", n_jobs=-1)
    nn.fit(X_train)
    dists, idxs = nn.kneighbors(X_query, return_distance=True)
    sims = np.clip(1.0 - dists, 0.0, None)
    w = np.power(sims + 1e-12, alpha)
    yN = ylog_train[idxs]
    num = (w * yN).sum(axis=1); den = w.sum(axis=1)
    return np.where(den>0, num/den, np.median(ylog_train))

# ---------------- OOF for embedding-kNN ----------------
bins = pd.qcut(ylog_sub, q=min(20, len(ylog_sub)), labels=False, duplicates="drop")
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_knn_log = np.zeros(len(ylog_sub), dtype="float32")
for f,(tr,va) in enumerate(skf.split(Xtr, bins),1):
    pred = knn_predict_emb(Xtr[tr], ylog_sub[tr], Xtr[va], K_NEIGHBORS, ALPHA)
    oof_knn_log[va] = pred
    np.save(OUT_DIR/"oof_knn_log_running.npy", oof_knn_log)
oof_knn_price_sub = np.expm1(oof_knn_log)
oof_knn_full = np.full(len(train), np.nan, dtype="float32"); oof_knn_full[tr_idx] = oof_knn_price_sub

# ---------------- TF-IDF + Ridge (OOF) ----------------
texts_full = train["__text__"].astype(str).values
def tfidf_ridge_oof(texts, y, folds=N_FOLDS):
    ylog = np.log1p(y)
    bins = pd.qcut(ylog, q=min(20,len(ylog)), labels=False, duplicates="drop")
    skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=SEED)
    oof = np.zeros(len(texts), dtype="float32")
    for f,(tr,va) in enumerate(skf.split(texts, bins),1):
        word = TfidfVectorizer(ngram_range=(1,2), min_df=3, max_features=WORD_MAX)
        char = TfidfVectorizer(analyzer="char", ngram_range=(3,5), min_df=3, max_features=CHAR_MAX)
        Xw_tr = word.fit_transform(texts[tr]); Xc_tr = char.fit_transform(texts[tr])
        X_tr  = hstack([Xw_tr,Xc_tr], format="csr")
        Xw_va = word.transform(texts[va]);  Xc_va = char.transform(texts[va])
        X_va  = hstack([Xw_va,Xc_va], format="csr")
        model = Ridge(alpha=RIDGE_ALPHA, random_state=SEED).fit(X_tr, np.log1p(y[tr]))
        oof[va] = model.predict(X_va)
    return oof
oof_tfidf_log = tfidf_ridge_oof(texts_full, y_full, N_FOLDS)
oof_tfidf_price = np.expm1(oof_tfidf_log)

# ---------------- Blend weight via OOF SMAPE ----------------
def smape(y, p):
    y, p = y.astype(float), p.astype(float)
    d=(np.abs(y)+np.abs(p))/2.0
    return 100*np.mean(np.where(d==0,0,np.abs(p-y)/d))

mask = np.isfinite(oof_knn_full) & np.isfinite(oof_tfidf_price)
ws = np.linspace(0,1,51)
best_w = min(ws, key=lambda w: smape(y_full[mask], w*oof_knn_full[mask] + (1-w)*oof_tfidf_price[mask]))
print(f"OOF SMAPE kNN:    {smape(y_full[mask], oof_knn_full[mask]):.3f}%")
print(f"OOF SMAPE TF-IDF: {smape(y_full[mask], oof_tfidf_price[mask]):.3f}%")
print("Best blend w (kNN share):", best_w)
with open(OUT_DIR/"blend_weight.json","w") as f:
    json.dump({"w_knn": float(best_w)}, f)

# ---------------- Fit FINAL TF-IDF+Ridge and SAVE ----------------
word = TfidfVectorizer(ngram_range=(1,2), min_df=3, max_features=WORD_MAX)
char = TfidfVectorizer(analyzer="char", ngram_range=(3,5), min_df=3, max_features=CHAR_MAX)
Xw = word.fit_transform(train["__text__"]); Xc = char.fit_transform(train["__text__"])
X_all = hstack([Xw,Xc], format="csr")
ridge = Ridge(alpha=RIDGE_ALPHA, random_state=SEED).fit(X_all, np.log1p(y_full))
joblib.dump(word,  MODEL_DIR/"word_vec.pkl")
joblib.dump(char,  MODEL_DIR/"char_vec.pkl")
joblib.dump(ridge, MODEL_DIR/"ridge.pkl")

# ---------------- Predict TEST (both heads) ----------------
test_knn_log   = knn_predict_emb(Xtr, ylog_sub, Xte, K_NEIGHBORS, ALPHA)
test_knn_price = np.expm1(test_knn_log)

Xw_te = word.transform(test["__text__"]); Xc_te = char.transform(test["__text__"])
X_te  = hstack([Xw_te,Xc_te], format="csr")
test_tfidf_price = np.expm1(ridge.predict(X_te))

test_price_blend = best_w*test_knn_price + (1-best_w)*test_tfidf_price

# ---------------- Save submissions & OOF (for global blend) ----------------
pd.DataFrame({"sample_id":test["sample_id"], "price":test_knn_price })\
  .to_csv(OUT_DIR/"submission_text_knn.csv", index=False)
pd.DataFrame({"sample_id":test["sample_id"], "price":test_tfidf_price})\
  .to_csv(OUT_DIR/"submission_text_tfidf.csv", index=False)
pd.DataFrame({"sample_id":test["sample_id"], "price":test_price_blend})\
  .to_csv(OUT_DIR/"submission_text_blend.csv", index=False)

np.save(OUT_DIR/"oof_text_knn_price.npy",   oof_knn_full)
np.save(OUT_DIR/"oof_text_tfidf_price.npy", oof_tfidf_price)

print("\nSaved:")
print("  ", OUT_DIR/"submission_text_blend.csv")
print("Models:")
print("  ", MODEL_DIR/"word_vec.pkl")
print("  ", MODEL_DIR/"char_vec.pkl")
print("  ", MODEL_DIR/"ridge.pkl")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 84.2 MB/s eta 0:00:00
Device: cuda Tesla T4
Encoding TRAIN...


Batches:   0%|          | 0/74 [00:00<?, ?it/s]

Encoding TEST ...


Batches:   0%|          | 0/74 [00:00<?, ?it/s]

OOF SMAPE kNN:    58.179%
OOF SMAPE TF-IDF: 52.798%
Best blend w (kNN share): 0.06

Saved:
   /content/drive/MyDrive/ml/text_best_cpu_v1/submission_text_blend.csv
Models:
   /content/drive/MyDrive/ml/text_best_cpu_v1/model/word_vec.pkl
   /content/drive/MyDrive/ml/text_best_cpu_v1/model/char_vec.pkl
   /content/drive/MyDrive/ml/text_best_cpu_v1/model/ridge.pkl


In [ ]:
# =============== TEXT-ONLY (No-FAISS): e5 embeddings + sklearn kNN + TF-IDF Ridge ===============
# • Fast, robust baseline with OOF SMAPE and auto-blended test preds
# • GPU (if available) ONLY speeds up embedding; kNN/TF-IDF/Ridge run on CPU cores
# • Checkpoints: caches embeddings and writes OOF/intermediate artifacts; safe to resume
# -----------------------------------------------------------------------------------------------
from pathlib import Path
CSV_DIR = Path("/content/drive/MyDrive/ml/ML challenge data/dataset")   # <-- edit if needed
OUT_DIR = Path("/content/drive/MyDrive/ml/text_best_cpu_v1")           # change per account when parallelizing
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Minimal deps (uncomment in Colab) ---
!pip -q install -U sentence-transformers scikit-learn scipy joblib

import os, gc, json, random, numpy as np, pandas as pd, joblib
from tqdm.auto import tqdm
import torch
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import NearestNeighbors
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from scipy.sparse import hstack

# ---------------- Device & threads ----------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE=="cuda" else "")
if DEVICE == "cpu":
    CORES = max(4, os.cpu_count() or 4)
    os.environ.update({
        "OMP_NUM_THREADS": str(CORES), "MKL_NUM_THREADS": str(CORES),
        "OPENBLAS_NUM_THREADS": str(CORES), "NUMEXPR_NUM_THREADS": str(CORES),
        "VECLIB_MAXIMUM_THREADS": str(CORES),
    })
    torch.set_num_threads(CORES); torch.set_num_interop_threads(max(1, CORES//2))
else:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

# ---------------- Config ----------------
SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
N_FOLDS=5
MODEL_NAME = "intfloat/e5-base-v2"       # strong, plug-and-play
TEXT_PREFIX = "passage: "                # e5 works best with a prefix
BATCH_ENC = 1024 if DEVICE=="cuda" else 256
K_NEIGHBORS=32
ALPHA=2.0                                # weight neighbors by (cos_sim)^ALPHA
WORD_MAX=200_000; CHAR_MAX=100_000
RIDGE_ALPHA=3.0

CACHE = OUT_DIR/"cache"; CACHE.mkdir(parents=True, exist_ok=True)
MODEL_DIR = OUT_DIR/"model"; MODEL_DIR.mkdir(parents=True, exist_ok=True)

# ---------------- Data ----------------
train = pd.read_csv(CSV_DIR/"train.csv")
test  = pd.read_csv(CSV_DIR/"test.csv")

CAND = ["catalog_content","description","product_description","title","name","product_name"]
def pick_text_col(df):
    low={c.lower():c for c in df.columns}
    for c in CAND:
        if c in low: return low[c]
    raise RuntimeError(f"No text column found; expected one of {CAND}")
TEXT_COL = pick_text_col(train)
assert TEXT_COL in test.columns, f"Test missing {TEXT_COL}"
train["__text__"] = train[TEXT_COL].fillna("").astype(str)
test["__text__"]  = test[TEXT_COL].fillna("").astype(str)
mask_tr = train["__text__"].str.strip()!=""

y_full = train["price"].values.astype(float)

# ---------------- Embeddings (cached) ----------------
TRN_EMB = CACHE/"train_emb.npy"; TST_EMB = CACHE/"test_emb.npy"; TRN_IDX = CACHE/"train_idx.npy"
if TRN_EMB.exists() and TST_EMB.exists() and TRN_IDX.exists():
    Xtr = np.load(TRN_EMB, mmap_mode="r"); Xte = np.load(TST_EMB, mmap_mode="r")
    tr_idx = np.load(TRN_IDX); print("Loaded cached embeddings:", Xtr.shape, Xte.shape)
else:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer(MODEL_NAME, device=DEVICE); model.max_seq_length = 256

    # Prefix EACH string, not the whole list
    tr_series = train.loc[mask_tr, "__text__"].fillna("").astype(str)
    te_series = test["__text__"].fillna("").astype(str)
    tr_texts = [f"{TEXT_PREFIX}{t}" for t in tr_series.tolist()]
    te_texts = [f"{TEXT_PREFIX}{t}" for t in te_series.tolist()]

    def encode(texts, path):
        embs = model.encode(
            texts,
            batch_size=BATCH_ENC,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=True
        ).astype("float32")
        np.save(path, embs)
        return embs

    print("Encoding TRAIN..."); Xtr = encode(tr_texts, TRN_EMB)
    print("Encoding TEST ...");  Xte  = encode(te_texts, TST_EMB)
    tr_idx = train.index[mask_tr].to_numpy(dtype=int); np.save(TRN_IDX, tr_idx)
    del model; gc.collect()

y_sub = y_full[tr_idx]; ylog_sub = np.log1p(y_sub)

# ---------------- kNN (cosine) helpers ----------------
def knn_predict_emb(X_train, ylog_train, X_query, k=K_NEIGHBORS, alpha=ALPHA):
    nn = NearestNeighbors(n_neighbors=k, metric="cosine", n_jobs=-1)
    nn.fit(X_train)
    dists, idxs = nn.kneighbors(X_query, return_distance=True)
    sims = np.clip(1.0 - dists, 0.0, None)
    w = np.power(sims + 1e-12, alpha)
    yN = ylog_train[idxs]
    num = (w * yN).sum(axis=1); den = w.sum(axis=1)
    return np.where(den>0, num/den, np.median(ylog_train))

# ---------------- OOF for embedding-kNN ----------------
bins = pd.qcut(ylog_sub, q=min(20, len(ylog_sub)), labels=False, duplicates="drop")
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_knn_log = np.zeros(len(ylog_sub), dtype="float32")
for f,(tr,va) in enumerate(skf.split(Xtr, bins),1):
    pred = knn_predict_emb(Xtr[tr], ylog_sub[tr], Xtr[va], K_NEIGHBORS, ALPHA)
    oof_knn_log[va] = pred
    np.save(OUT_DIR/"oof_knn_log_running.npy", oof_knn_log)
oof_knn_price_sub = np.expm1(oof_knn_log)
oof_knn_full = np.full(len(train), np.nan, dtype="float32"); oof_knn_full[tr_idx] = oof_knn_price_sub

# ---------------- TF-IDF + Ridge (OOF) ----------------
texts_full = train["__text__"].astype(str).values
def tfidf_ridge_oof(texts, y, folds=N_FOLDS):
    ylog = np.log1p(y)
    bins = pd.qcut(ylog, q=min(20,len(ylog)), labels=False, duplicates="drop")
    skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=SEED)
    oof = np.zeros(len(texts), dtype="float32")
    for f,(tr,va) in enumerate(skf.split(texts, bins),1):
        word = TfidfVectorizer(ngram_range=(1,2), min_df=3, max_features=WORD_MAX)
        char = TfidfVectorizer(analyzer="char", ngram_range=(3,5), min_df=3, max_features=CHAR_MAX)
        Xw_tr = word.fit_transform(texts[tr]); Xc_tr = char.fit_transform(texts[tr])
        X_tr  = hstack([Xw_tr,Xc_tr], format="csr")
        Xw_va = word.transform(texts[va]);  Xc_va = char.transform(texts[va])
        X_va  = hstack([Xw_va,Xc_va], format="csr")
        model = Ridge(alpha=RIDGE_ALPHA, random_state=SEED).fit(X_tr, np.log1p(y[tr]))
        oof[va] = model.predict(X_va)
    return oof
oof_tfidf_log = tfidf_ridge_oof(texts_full, y_full, N_FOLDS)
oof_tfidf_price = np.expm1(oof_tfidf_log)

# ---------------- Blend weight via OOF SMAPE ----------------
def smape(y, p):
    y, p = y.astype(float), p.astype(float)
    d=(np.abs(y)+np.abs(p))/2.0
    return 100*np.mean(np.where(d==0,0,np.abs(p-y)/d))

mask = np.isfinite(oof_knn_full) & np.isfinite(oof_tfidf_price)
ws = np.linspace(0,1,51)
best_w = min(ws, key=lambda w: smape(y_full[mask], w*oof_knn_full[mask] + (1-w)*oof_tfidf_price[mask]))
print(f"OOF SMAPE kNN:    {smape(y_full[mask], oof_knn_full[mask]):.3f}%")
print(f"OOF SMAPE TF-IDF: {smape(y_full[mask], oof_tfidf_price[mask]):.3f}%")
print("Best blend w (kNN share):", best_w)
with open(OUT_DIR/"blend_weight.json","w") as f:
    json.dump({"w_knn": float(best_w)}, f)

# ---------------- Fit FINAL TF-IDF+Ridge and SAVE ----------------
word = TfidfVectorizer(ngram_range=(1,2), min_df=3, max_features=WORD_MAX)
char = TfidfVectorizer(analyzer="char", ngram_range=(3,5), min_df=3, max_features=CHAR_MAX)
Xw = word.fit_transform(train["__text__"]); Xc = char.fit_transform(train["__text__"])
X_all = hstack([Xw,Xc], format="csr")
ridge = Ridge(alpha=RIDGE_ALPHA, random_state=SEED).fit(X_all, np.log1p(y_full))
joblib.dump(word,  MODEL_DIR/"word_vec.pkl")
joblib.dump(char,  MODEL_DIR/"char_vec.pkl")
joblib.dump(ridge, MODEL_DIR/"ridge.pkl")

# ---------------- Predict TEST (both heads) ----------------
test_knn_log   = knn_predict_emb(Xtr, ylog_sub, Xte, K_NEIGHBORS, ALPHA)
test_knn_price = np.expm1(test_knn_log)

Xw_te = word.transform(test["__text__"]); Xc_te = char.transform(test["__text__"])
X_te  = hstack([Xw_te,Xc_te], format="csr")
test_tfidf_price = np.expm1(ridge.predict(X_te))

test_price_blend = best_w*test_knn_price + (1-best_w)*test_tfidf_price

# ---------------- Save submissions & OOF (for global blend) ----------------
pd.DataFrame({"sample_id":test["sample_id"], "price":test_knn_price })\
  .to_csv(OUT_DIR/"submission_text_knn.csv", index=False)
pd.DataFrame({"sample_id":test["sample_id"], "price":test_tfidf_price})\
  .to_csv(OUT_DIR/"submission_text_tfidf.csv", index=False)
pd.DataFrame({"sample_id":test["sample_id"], "price":test_price_blend})\
  .to_csv(OUT_DIR/"submission_text_blend.csv", index=False)

np.save(OUT_DIR/"oof_text_knn_price.npy",   oof_knn_full)
np.save(OUT_DIR/"oof_text_tfidf_price.npy", oof_tfidf_price)

print("\nSaved:")
print("  ", OUT_DIR/"submission_text_blend.csv")
print("Models:")
print("  ", MODEL_DIR/"word_vec.pkl")
print("  ", MODEL_DIR/"char_vec.pkl")
print("  ", MODEL_DIR/"ridge.pkl")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 84.2 MB/s eta 0:00:00
Device: cuda Tesla T4
Encoding TRAIN...


Batches:   0%|          | 0/74 [00:00<?, ?it/s]

Encoding TEST ...


Batches:   0%|          | 0/74 [00:00<?, ?it/s]

OOF SMAPE kNN:    58.179%
OOF SMAPE TF-IDF: 52.798%
Best blend w (kNN share): 0.06

Saved:
   /content/drive/MyDrive/ml/text_best_cpu_v1/submission_text_blend.csv
Models:
   /content/drive/MyDrive/ml/text_best_cpu_v1/model/word_vec.pkl
   /content/drive/MyDrive/ml/text_best_cpu_v1/model/char_vec.pkl
   /content/drive/MyDrive/ml/text_best_cpu_v1/model/ridge.pkl


In [5]:
# Colab-safe upgrades; keeps NumPy as-is and adds the libs we need
!pip -q install -U optuna lightgbm xgboost catboost --no-input


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 9.4 MB/s eta 0:00:00


In [6]:
import gc, json, numpy as np, pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
from xgboost import XGBRegressor
from catboost import CatBoostRegressor, Pool
import optuna
from optuna.pruners import MedianPruner

SEED = globals().get("SEED", 42)
N_FOLDS = globals().get("N_FOLDS", 5)

def smape(y, p):
    y, p = y.astype(float), p.astype(float)
    d = (np.abs(y) + np.abs(p)) / 2.0
    return 100 * np.mean(np.where(d==0, 0, np.abs(p - y) / d))

def make_bins_for_cv(y_log, q=20):
    return pd.qcut(y_log, q=min(q, len(y_log)), labels=False, duplicates="drop")

def ensure_embeddings():
    # use already-computed e5 embeddings or load from cache
    global Xtr, Xte, tr_idx, y_full
    TRN_EMB = CACHE/"train_emb.npy"
    TST_EMB = CACHE/"test_emb.npy"
    TRN_IDX = CACHE/"train_idx.npy"
    if "Xtr" not in globals() or "Xte" not in globals():
        Xtr = np.load(TRN_EMB, mmap_mode="r")
        Xte = np.load(TST_EMB, mmap_mode="r")
        tr_idx = np.load(TRN_IDX)
    return Xtr, Xte, tr_idx

def ensure_tfidf_mats():
    # use existing TF-IDF matrices or rebuild from fitted vectorizers
    global X_all, X_te, word, char, train, test
    if "X_all" not in globals() or "X_te" not in globals():
        from scipy.sparse import hstack
        Xw = word.fit_transform(train["__text__"])
        Xc = char.fit_transform(train["__text__"])
        X_all = hstack([Xw, Xc], format="csr")
        Xw_te = word.transform(test["__text__"])
        Xc_te = char.transform(test["__text__"])
        X_te = hstack([Xw_te, Xc_te], format="csr")
    return X_all, X_te

rng = np.random.RandomState(SEED)

In [1]:
# ==== BOOTSTRAP AFTER RESTART (run once) ======================================
# Rebuild paths, data, vectorizers, TF-IDF mats, Ridge, embeddings, OOFs & test preds

import os, gc, json, numpy as np, pandas as pd, joblib
from pathlib import Path

# ---- Paths & constants (match your baseline) ----
CSV_DIR  = globals().get("CSV_DIR", Path("/content/drive/MyDrive/ml/ML challenge data/dataset"))
OUT_DIR  = globals().get("OUT_DIR", Path("/content/drive/MyDrive/ml/text_best_cpu_v1"))
CACHE    = globals().get("CACHE", OUT_DIR / "cache")
MODEL_DIR= globals().get("MODEL_DIR", OUT_DIR / "model")
for p in [OUT_DIR, CACHE, MODEL_DIR]: p.mkdir(parents=True, exist_ok=True)

SEED      = globals().get("SEED", 42)
N_FOLDS   = globals().get("N_FOLDS", 5)
WORD_MAX  = globals().get("WORD_MAX", 200_000)
CHAR_MAX  = globals().get("CHAR_MAX", 100_000)
K_NEIGHBORS = globals().get("K_NEIGHBORS", 32)
ALPHA     = globals().get("ALPHA", 2.0)

# ---- Data & text column ----
if "train" not in globals() or "test" not in globals():
    train = pd.read_csv(CSV_DIR / "train.csv")
    test  = pd.read_csv(CSV_DIR / "test.csv")

CAND = ["catalog_content","description","product_description","title","name","product_name"]
def pick_text_col(df):
    low={c.lower():c for c in df.columns}
    for c in CAND:
        if c in low: return low[c]
    raise RuntimeError(f"No text column found; expected one of {CAND}")

if "__text__" not in train.columns or "__text__" not in test.columns:
    TEXT_COL = pick_text_col(train)
    assert TEXT_COL in test.columns, f"Test missing {TEXT_COL}"
    train["__text__"] = train[TEXT_COL].fillna("").astype(str)
    test["__text__"]  = test[TEXT_COL].fillna("").astype(str)

y_full = train["price"].values.astype(float)

# ---- Vectorizers & TF-IDF matrices ----
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

w_path = MODEL_DIR / "word_vec.pkl"
c_path = MODEL_DIR / "char_vec.pkl"
try:
    if w_path.exists() and c_path.exists():
        word = joblib.load(w_path); char = joblib.load(c_path)
    else:
        raise FileNotFoundError
except Exception:
    word = TfidfVectorizer(ngram_range=(1,2), min_df=3, max_features=WORD_MAX)
    char = TfidfVectorizer(analyzer="char", ngram_range=(3,5), min_df=3, max_features=CHAR_MAX)
    # Fit below

# Build matrices (fit if needed)
try:
    Xw = word.transform(train["__text__"])
    Xc = char.transform(train["__text__"])
except Exception:
    Xw = word.fit_transform(train["__text__"])
    Xc = char.fit_transform(train["__text__"])
    try:
        joblib.dump(word, w_path); joblib.dump(char, c_path)
    except Exception:
        pass

X_all = hstack([Xw, Xc], format="csr")
Xw_te = word.transform(test["__text__"])
Xc_te = char.transform(test["__text__"])
X_te  = hstack([Xw_te, Xc_te], format="csr")

# ---- Ridge head (fit or load) + test_tfidf_price ----
from sklearn.linear_model import Ridge
ridge_path = MODEL_DIR / "ridge.pkl"
try:
    ridge = joblib.load(ridge_path)
except Exception:
    ridge = Ridge(alpha=3.0, random_state=SEED).fit(X_all, np.log1p(y_full))
    try: joblib.dump(ridge, ridge_path)
    except Exception: pass
test_tfidf_price = np.expm1(ridge.predict(X_te))

# ---- Try to load OOF TF-IDF from disk; else recompute quickly ----
oof_tfidf_path = OUT_DIR / "oof_text_tfidf_price.npy"
if oof_tfidf_path.exists():
    oof_tfidf_price = np.load(oof_tfidf_path)
else:
    # Recompute OOF TF-IDF Ridge
    from sklearn.model_selection import StratifiedKFold
    def tfidf_ridge_oof(texts, y, folds=N_FOLDS):
        ylog = np.log1p(y)
        bins = pd.qcut(ylog, q=min(20,len(ylog)), labels=False, duplicates="drop")
        skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=SEED)
        oof = np.zeros(len(texts), dtype="float32")
        for f,(tr,va) in enumerate(skf.split(np.arange(len(texts)), bins),1):
            wv = TfidfVectorizer(ngram_range=(1,2), min_df=3, max_features=WORD_MAX)
            cv = TfidfVectorizer(analyzer="char", ngram_range=(3,5), min_df=3, max_features=CHAR_MAX)
            Xw_tr = wv.fit_transform(texts[tr]); Xc_tr = cv.fit_transform(texts[tr])
            X_tr  = hstack([Xw_tr,Xc_tr], format="csr")
            Xw_va = wv.transform(texts[va]);  Xc_va = cv.transform(texts[va])
            X_va  = hstack([Xw_va,Xc_va], format="csr")
            m = Ridge(alpha=3.0, random_state=SEED).fit(X_tr, np.log1p(y[tr]))
            oof[va] = m.predict(X_va)
        return np.expm1(oof)
    oof_tfidf_price = tfidf_ridge_oof(train["__text__"].values.astype(str), y_full, N_FOLDS)
    np.save(oof_tfidf_path, oof_tfidf_price)

# ---- Embeddings & kNN preds (load from cache; compute if needed) ----
TRN_EMB = CACHE/"train_emb.npy"; TST_EMB = CACHE/"test_emb.npy"; TRN_IDX = CACHE/"train_idx.npy"
if not (TRN_EMB.exists() and TST_EMB.exists() and TRN_IDX.exists()):
    raise RuntimeError("Missing cached embeddings. Re-run the embedding cell first.")
Xtr = np.load(TRN_EMB, mmap_mode="r"); Xte = np.load(TST_EMB, mmap_mode="r"); tr_idx = np.load(TRN_IDX)
ylog_sub = np.log1p(y_full[tr_idx])

# test kNN price
knn_csv = OUT_DIR / "submission_text_knn.csv"
if knn_csv.exists():
    test_knn_price = pd.read_csv(knn_csv)["price"].values
else:
    # recompute quickly
    from sklearn.neighbors import NearestNeighbors
    def knn_predict_emb(X_train, ylog_train, X_query, k=K_NEIGHBORS, alpha=ALPHA):
        nn = NearestNeighbors(n_neighbors=k, metric="cosine", n_jobs=-1).fit(X_train)
        dists, idxs = nn.kneighbors(X_query, return_distance=True)
        sims = np.clip(1.0 - dists, 0.0, None)
        w = np.power(sims + 1e-12, alpha)
        yN = ylog_train[idxs]
        num = (w * yN).sum(axis=1); den = w.sum(axis=1)
        return np.expm1(np.where(den>0, num/den, np.median(ylog_train)))
    test_knn_price = knn_predict_emb(Xtr, ylog_sub, Xte, K_NEIGHBORS, ALPHA)

# OOF kNN
oof_knn_path = OUT_DIR / "oof_text_knn_price.npy"
if oof_knn_path.exists():
    oof_knn_full = np.load(oof_knn_path)
else:
    # Recompute OOF kNN via CV on embeddings subset
    from sklearn.model_selection import StratifiedKFold
    from sklearn.neighbors import NearestNeighbors
    def knn_predict_emb_log(X_train, ylog_train, X_query, k=K_NEIGHBORS, alpha=ALPHA):
        nn = NearestNeighbors(n_neighbors=k, metric="cosine", n_jobs=-1).fit(X_train)
        dists, idxs = nn.kneighbors(X_query, return_distance=True)
        sims = np.clip(1.0 - dists, 0.0, None)
        w = np.power(sims + 1e-12, alpha)
        yN = ylog_train[idxs]
        num = (w * yN).sum(axis=1); den = w.sum(axis=1)
        return np.where(den>0, num/den, np.median(ylog_train))

    bins = pd.qcut(ylog_sub, q=min(20, len(ylog_sub)), labels=False, duplicates="drop")
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    oof_knn_log_sub = np.zeros(len(ylog_sub), dtype="float32")
    for f,(tr,va) in enumerate(skf.split(Xtr, bins),1):
        oof_knn_log_sub[va] = knn_predict_emb_log(Xtr[tr], ylog_sub[tr], Xtr[va], K_NEIGHBORS, ALPHA)
    oof_knn_full = np.full(len(train), np.nan, dtype="float32")
    oof_knn_full[tr_idx] = np.expm1(oof_knn_log_sub)
    np.save(oof_knn_path, oof_knn_full)

# ---- Quick sanity prints ----
def smape(y, p):
    y, p = y.astype(float), p.astype(float)
    d=(np.abs(y)+np.abs(p))/2.0
    return 100*np.mean(np.where(d==0,0,np.abs(p-y)/d))

print("Ready. Shapes:")
print("  X_all:", X_all.shape, "| X_te:", X_te.shape)
print("  e5 train/test:", Xtr.shape, Xte.shape, "| subset size:", len(tr_idx))
print("OOF SMAPEs now available:")
print("  TF-IDF Ridge :", f"{smape(y_full, oof_tfidf_price):.3f}%")
print("  e5 kNN       :", f"{smape(y_full, oof_knn_full):.3f}%")


Ready. Shapes:
  X_all: (75000, 300000) | X_te: (75000, 300000)
  e5 train/test: (75000, 768) (75000, 768) | subset size: 75000
OOF SMAPEs now available:
  TF-IDF Ridge : 52.798%
  e5 kNN       : 58.179%


In [20]:
# Ensure necessary variables are available or re-calculated if needed
# This assumes the previous cells that calculate these are available in the notebook state.
# If not, the bootstrap cell (3aIq4ZA-i_7J) should be run first.

# Re-calculate stacked predictions
from sklearn.linear_model import Ridge

# Build test meta features in the SAME order
test_feat_list_log = [
    np.log1p(test_tfidf_price),
    np.log1p(test_knn_price),
    np.load(OUT_DIR/"test_lgb_svd_log.npy"),
    np.load(OUT_DIR/"test_xgb_svd_log.npy"),
    np.load(OUT_DIR/"test_lgb_e5_log.npy"),
    np.load(OUT_DIR/"test_cat_e5_log.npy"),
]
T = np.vstack(test_feat_list_log).T

# Load the fitted meta Ridge model (assuming it was saved)
meta_model_path = OUT_DIR / "meta_ridge_model.pkl"
try:
    meta = joblib.load(meta_model_path)
except FileNotFoundError:
    print("Meta Ridge model not found. Re-fitting the meta model...")
    # Rebuild meta features from LOG OOFs to re-fit the meta model
    feat_list_log = [
        np.log1p(oof_tfidf_price),
        np.log1p(oof_knn_full),
        oof_lgb_svd,
        oof_xgb_svd,
        oof_lgb_e5,
        oof_cat_e5
    ]
    F = np.vstack(feat_list_log).T
    mask = np.all(np.isfinite(F), axis=1)
    X_meta = F[mask]
    y_meta = np.log1p(y_full[mask]) # Use y_log[mask] if y_log is available and masked correctly
    meta = Ridge(alpha=1.0, random_state=SEED).fit(X_meta, y_meta)
    # Save the refitted meta model
    joblib.dump(meta, meta_model_path)


# Final test prediction
test_price_stack = np.expm1(meta.predict(T))

# Create DataFrame for submission
submission_df = pd.DataFrame({"sample_id": test["sample_id"], "price": test_price_stack})

# Display the head of the submission DataFrame
print("Head of the stacked predictions:")
display(submission_df.head())

# Save to CSV
sub_path = OUT_DIR/"submission_text_stack.csv"
submission_df.to_csv(sub_path, index=False)

print(f"Stacked predictions saved to: {sub_path}")

# Verify existence after saving
if os.path.exists(sub_path):
    print(f"Confirmed: The file {sub_path} now exists.")
else:
    print(f"Warning: The file {sub_path} still does not exist after attempting to save.")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/ml/text_best_cpu_v1/test_lgb_svd_log.npy'

In [21]:
# ==== META STACK: kNN + TF-IDF (+ blend as a feature) ====
from pathlib import Path
import os, json, numpy as np, pandas as pd, joblib
from sklearn.linear_model import Ridge

# --- Paths
CSV_DIR = Path("/content/drive/MyDrive/ml/ML challenge data/dataset")
OUT_DIR = Path("/content/drive/MyDrive/ml/text_best_cpu_v1")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Load base TRAIN data & targets
train = pd.read_csv(CSV_DIR/"train.csv")
test  = pd.read_csv(CSV_DIR/"test.csv")
y_full = train["price"].to_numpy(float)

# --- Load OOF preds (price space)
oof_knn_path   = OUT_DIR/"oof_text_knn_price.npy"
oof_tfidf_path = OUT_DIR/"oof_text_tfidf_price.npy"
assert oof_knn_path.exists() and oof_tfidf_path.exists(), "Run the text baseline first to produce OOF arrays."
oof_knn   = np.load(oof_knn_path)
oof_tfidf = np.load(oof_tfidf_path)

# --- Load test preds (price space), align by sample_id
def load_sub(path):
    df = pd.read_csv(path)
    id_col = "sample_id" if "sample_id" in df.columns else df.columns[0]
    pr_col = "price" if "price" in df.columns else df.columns[-1]
    m = test[[id_col]].merge(df[[id_col, pr_col]], on=id_col, how="left")
    return m[pr_col].to_numpy(float)

test_knn   = load_sub(OUT_DIR/"submission_text_knn.csv")
test_tfidf = load_sub(OUT_DIR/"submission_text_tfidf.csv")

# --- Optional: include the previous blend as a third feature
def smape(y, p):
    y, p = y.astype(float), p.astype(float)
    d = (np.abs(y) + np.abs(p)) / 2.0
    return 100*np.mean(np.where(d==0, 0, np.abs(p - y) / d))

w_json = OUT_DIR/"blend_weight.json"
if w_json.exists():
    w_knn = float(json.load(open(w_json))["w_knn"])
else:
    # fallback: small grid on OOF
    ws = np.linspace(0, 1, 51)
    w_knn = min(ws, key=lambda w: smape(y_full, w*oof_knn + (1-w)*oof_tfidf))
oof_blend   = w_knn*oof_knn + (1.0 - w_knn)*oof_tfidf
test_blend  = w_knn*test_knn + (1.0 - w_knn)*test_tfidf

# --- Build META features (log domain is stabler)
F = np.vstack([
    np.log1p(oof_knn),
    np.log1p(oof_tfidf),
    np.log1p(oof_blend),
]).T
T = np.vstack([
    np.log1p(test_knn),
    np.log1p(test_tfidf),
    np.log1p(test_blend),
]).T

# Safety masks
mask = np.all(np.isfinite(F), axis=1)
F = F[mask]
y_log = np.log1p(y_full[mask])

# --- Fit meta Ridge and save it
meta = Ridge(alpha=1.0, random_state=42).fit(F, y_log)
joblib.dump(meta, OUT_DIR/"meta_ridge_model.pkl")

# --- Predict test and write submission
pred_log = meta.predict(T)
pred = np.expm1(pred_log)

sub = pd.DataFrame({"sample_id": test["sample_id"], "price": pred})
save_path = OUT_DIR/"submission_text_stack.csv"
sub.to_csv(save_path, index=False)

print("Meta features used:", F.shape[1], "| Train rows:", F.shape[0], "| Test rows:", T.shape[0])
print("Blend weight (kNN share):", round(w_knn, 4))
print("Head:\n", sub.head())
print("Saved:", save_path)
print("Exists?", os.path.exists(save_path))


Meta features used: 3 | Train rows: 75000 | Test rows: 75000
Blend weight (kNN share): 0.06
Head:
    sample_id      price
0     100179  17.511362
1     245611  15.044946
2     146263  22.219650
3      95658   4.881846
4      36806  36.777796
Saved: /content/drive/MyDrive/ml/text_best_cpu_v1/submission_text_stack.csv
Exists? True


In [7]:
from scipy.sparse import issparse

X_all, X_te = ensure_tfidf_mats()

SVD_DIM = 512  # 256–1024 works; 512 is a sweet spot for speed/accuracy
svd = TruncatedSVD(n_components=SVD_DIM, random_state=SEED)
X_svd = svd.fit_transform(X_all)
X_svd_te = svd.transform(X_te)

print("SVD shapes:", X_svd.shape, X_svd_te.shape)


SVD shapes: (75000, 512) (75000, 512)


In [8]:
Xtr, Xte, tr_idx = ensure_embeddings()
y_full = train["price"].values.astype(float)
y_log = np.log1p(y_full)
y_log_sub = y_log[tr_idx]

print("Emb shapes:", Xtr.shape, Xte.shape, "| subset:", len(tr_idx))


Emb shapes: (75000, 768) (75000, 768) | subset: 75000


In [9]:
def cv_lgbm_dense(X, y_log, folds, params, early_stop=100):
    oof = np.zeros(len(y_log), dtype=float)
    models = []
    skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=SEED)
    bins = make_bins_for_cv(y_log)
    for fold, (tr, va) in enumerate(skf.split(X, bins), 1):
        m = LGBMRegressor(
            n_estimators=5000,
            random_state=SEED,
            **params
        )
        m.fit(
            X[tr], y_log[tr],
            eval_set=[(X[va], y_log[va])],
            eval_metric="rmse",
            callbacks=[early_stopping(early_stop, verbose=False), log_evaluation(period=0)]
        )
        oof[va] = m.predict(X[va], raw_score=False)
        models.append(m)
        gc.collect()
    return oof, models

def cv_xgb_dense(X, y_log, folds, params, early_stop=100):
    oof = np.zeros(len(y_log), dtype=float)
    models = []
    skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=SEED)
    bins = make_bins_for_cv(y_log)
    for fold, (tr, va) in enumerate(skf.split(X, bins), 1):
        m = XGBRegressor(
            n_estimators=20000,
            random_state=SEED,
            **params
        )
        m.fit(
            X[tr], y_log[tr],
            eval_set=[(X[va], y_log[va])],
            verbose=False,
            early_stopping_rounds=early_stop
        )
        oof[va] = m.predict(X[va])
        models.append(m)
        gc.collect()
    return oof, models

def cv_cat_dense(X, y_log, folds, params, early_stop=100):
    oof = np.zeros(len(y_log), dtype=float)
    models = []
    skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=SEED)
    bins = make_bins_for_cv(y_log)
    for fold, (tr, va) in enumerate(skf.split(X, bins), 1):
        train_pool = Pool(X[tr], y_log[tr])
        valid_pool = Pool(X[va], y_log[va])
        m = CatBoostRegressor(
            iterations=20000,
            random_seed=SEED,
            loss_function="RMSE",
            eval_metric="RMSE",
            od_type="Iter",
            od_wait=early_stop,
            **params
        )
        m.fit(train_pool, eval_set=valid_pool, verbose=False)
        oof[va] = m.predict(valid_pool)
        models.append(m)
        gc.collect()
    return oof, models


In [10]:
!pip install optuna

In [13]:
# --- SAVE CURRENT SVD OUTPUTS (run once if X_svd already exists) ---
import numpy as np, json, joblib
from pathlib import Path

SVD_DIM = X_svd.shape[1]  # infer from your current result
SVD_DIR = OUT_DIR / "svd"; SVD_DIR.mkdir(parents=True, exist_ok=True)
SVD_TRN = SVD_DIR / f"X_svd_{SVD_DIM}.npy"
SVD_TST = SVD_DIR / f"X_svd_te_{SVD_DIM}.npy"
SVD_OBJ = SVD_DIR / f"svd_{SVD_DIM}.joblib"
SVD_META = SVD_DIR / f"meta_{SVD_DIM}.json"

# Save arrays (float32 reduces disk)
np.save(SVD_TRN, X_svd.astype(np.float32, copy=False))
np.save(SVD_TST, X_svd_te.astype(np.float32, copy=False))

# Save fitted SVD object if you have it in a variable named `svd` (optional)
try:
    import joblib
    joblib.dump(svd, SVD_OBJ)
except NameError:
    pass  # you can skip if you didn't keep the SVD object

# Save meta to detect mismatches later
with open(SVD_META, "w") as f:
    json.dump({
        "dim": int(SVD_DIM),
        "X_all_shape": tuple(X_all.shape),
        "X_te_shape": tuple(X_te.shape),
    }, f)

print("Saved SVD:")
print(" ", SVD_TRN)
print(" ", SVD_TST)
print(" ", SVD_OBJ.exists() and SVD_OBJ or "(svd object not saved)")


Saved SVD:
  /content/drive/MyDrive/ml/text_best_cpu_v1/svd/X_svd_512.npy
  /content/drive/MyDrive/ml/text_best_cpu_v1/svd/X_svd_te_512.npy
  /content/drive/MyDrive/ml/text_best_cpu_v1/svd/svd_512.joblib


In [14]:
# --- ENSURE/LOAD CACHED SVD (idempotent) ---
import json, gc, numpy as np, joblib
from pathlib import Path
from sklearn.decomposition import TruncatedSVD

SVD_DIM = 512  # <- set your preferred dim (use 512 or whatever you used before)
SVD_DIR = OUT_DIR / "svd"; SVD_DIR.mkdir(parents=True, exist_ok=True)
SVD_TRN = SVD_DIR / f"X_svd_{SVD_DIM}.npy"
SVD_TST = SVD_DIR / f"X_svd_te_{SVD_DIM}.npy"
SVD_OBJ = SVD_DIR / f"svd_{SVD_DIM}.joblib"
SVD_META = SVD_DIR / f"meta_{SVD_DIM}.json"

def ensure_svd(X_all, X_te, dim=SVD_DIM, seed=SEED, force_recompute=False, dtype=np.float32):
    if (not force_recompute) and SVD_TRN.exists() and SVD_TST.exists() and SVD_META.exists():
        try:
            meta = json.load(open(SVD_META))
            if (meta.get("dim")==dim and
                tuple(meta.get("X_all_shape",())) == tuple(X_all.shape) and
                tuple(meta.get("X_te_shape",())) == tuple(X_te.shape)):
                X_svd = np.load(SVD_TRN, mmap_mode="r")
                X_svd_te = np.load(SVD_TST, mmap_mode="r")
                svd_obj = joblib.load(SVD_OBJ) if SVD_OBJ.exists() else None
                print(f"[SVD] Loaded cache ({dim}) ->", X_svd.shape, X_svd_te.shape)
                return X_svd, X_svd_te, svd_obj
        except Exception:
            pass
        print("[SVD] Cache mismatch/corrupt → recomputing…")

    svd_obj = TruncatedSVD(n_components=dim, random_state=seed)
    X_svd = svd_obj.fit_transform(X_all).astype(dtype, copy=False)
    X_svd_te = svd_obj.transform(X_te).astype(dtype, copy=False)
    np.save(SVD_TRN, X_svd); np.save(SVD_TST, X_svd_te); joblib.dump(svd_obj, SVD_OBJ)
    json.dump({"dim": dim, "X_all_shape": X_all.shape, "X_te_shape": X_te.shape,
               "explained_variance_sum": float(np.sum(svd_obj.explained_variance_ratio_))},
              open(SVD_META, "w"))
    print(f"[SVD] Fit+saved ({dim}) | EVR sum={np.sum(svd_obj.explained_variance_ratio_):.4f}")
    gc.collect()
    return X_svd, X_svd_te, svd_obj

# Use it:
X_svd, X_svd_te, svd = ensure_svd(X_all, X_te, dim=SVD_DIM)


[SVD] Loaded cache (512) -> (75000, 512) (75000, 512)


In [15]:
# ===== Randomized Search (CPU) – setup =====
import os, numpy as np, json
from numpy.random import default_rng

# Force CPU & set threads (tune if Colab shows 2 vCPUs)
CORES = max(2, os.cpu_count() or 2)
os.environ.update({
    "CUDA_VISIBLE_DEVICES": "",
    "OMP_NUM_THREADS": str(CORES),
    "OPENBLAS_NUM_THREADS": str(CORES),
    "MKL_NUM_THREADS": str(CORES),
    "VECLIB_MAXIMUM_THREADS": str(CORES),
    "NUMEXPR_NUM_THREADS": str(CORES),
})
THREADS_PER_MODEL = CORES   # safe: 1 trial at a time

rng = default_rng(42)

# ---- parameter spaces (simple but effective) ----
space_lgb = {
    "lr":        ("log_uniform", 0.01, 0.2),
    "num_leaves":("int", 16, 256, "log"),
    "min_data_in_leaf":("int", 10, 200, None),
    "feature_fraction":("uniform", 0.6, 1.0),
    "bagging_fraction":("uniform", 0.6, 1.0),
    "bagging_freq":("int", 0, 10, None),
    "lambda_l1": ("uniform", 0.0, 5.0),
    "lambda_l2": ("uniform", 0.0, 5.0),
}
space_xgb = {
    "max_depth":      ("int", 3, 10, None),
    "min_child_weight":("log_uniform", 1e-2, 20.0),
    "subsample":      ("uniform", 0.6, 1.0),
    "colsample_bytree":("uniform", 0.6, 1.0),
    "eta":            ("log_uniform", 0.01, 0.3),
    "reg_alpha":      ("uniform", 0.0, 2.0),
    "reg_lambda":     ("uniform", 0.0, 2.0),
}
space_cat = {
    "depth":             ("int", 4, 10, None),
    "learning_rate":     ("log_uniform", 0.01, 0.3),
    "l2_leaf_reg":       ("uniform", 1.0, 20.0),
    "bagging_temperature":("uniform", 0.0, 5.0),
}

def sample(space):
    out = {}
    for k, spec in space.items():
        kind = spec[0]
        if kind == "uniform":
            lo, hi = spec[1], spec[2]
            out[k] = float(rng.uniform(lo, hi))
        elif kind == "log_uniform":
            lo, hi = np.log(spec[1]), np.log(spec[2])
            out[k] = float(np.exp(rng.uniform(lo, hi)))
        elif kind == "int":
            lo, hi, mode = spec[1], spec[2], spec[3]
            if mode == "log":
                out[k] = int(np.exp(rng.uniform(np.log(lo), np.log(hi))))
            else:
                out[k] = int(rng.integers(lo, hi+1))
        else:
            raise ValueError(kind)
    return out


In [16]:
from tqdm.auto import tqdm

# ==== generic RS on our CV helpers (keeps early-stopping) ====
def rs_cv(cv_fn, X, y_log, base_params, space, n_iter, early_stop, name):
    best_score = 1e9
    best_params = None
    for _ in tqdm(range(n_iter), desc=f"RS {name}"):
        p = sample(space)
        params = {**base_params, **p}
        oof_log, _ = cv_fn(X, y_log, N_FOLDS, params, early_stop=early_stop)
        score = smape(np.expm1(y_log), np.expm1(oof_log))
        if score < best_score:
            best_score, best_params = score, params
    print(f"[{name}] best SMAPE={best_score:.3f}% | params={best_params}")
    return best_params

# ---- base params forcing CPU threads ----
base_lgb = {"objective":"regression", "verbosity":-1, "n_jobs":THREADS_PER_MODEL}
base_xgb = {"tree_method":"hist", "predictor":"cpu_predictor", "n_jobs":THREADS_PER_MODEL}
base_cat = {"task_type":"CPU", "thread_count":THREADS_PER_MODEL, "loss_function":"RMSE"}

# ---- fast settings; increase n_iter later if needed ----
N_ITER_LGB_SVD = 15
N_ITER_XGB_SVD = 15
N_ITER_LGB_E5  = 12
N_ITER_CAT_E5  = 12

# LGBM on SVD(TF-IDF)
best_lgb_svd = rs_cv(cv_lgbm_dense, X_svd, y_log, base_lgb, space_lgb, N_ITER_LGB_SVD, early_stop=100, name="LGB-SVD")

# XGB on SVD(TF-IDF)
best_xgb_svd = rs_cv(cv_xgb_dense, X_svd, y_log, base_xgb, space_xgb, N_ITER_XGB_SVD, early_stop=200, name="XGB-SVD")

# LGBM on e5 (subset)
best_lgb_e5  = rs_cv(cv_lgbm_dense, Xtr, y_log_sub, base_lgb, space_lgb, N_ITER_LGB_E5, early_stop=100, name="LGB-e5")

# CatBoost on e5 (subset)
best_cat_e5  = rs_cv(cv_cat_dense, Xtr, y_log_sub, base_cat, space_cat, N_ITER_CAT_E5, early_stop=200, name="CAT-e5")

# Persist
json.dump({
    "lgb_svd": best_lgb_svd,
    "xgb_svd": best_xgb_svd,
    "lgb_e5": best_lgb_e5,
    "cat_e5": best_cat_e5,
}, open(OUT_DIR/"randomsearch_text_best.json","w"))

print("Saved best params →", OUT_DIR/"randomsearch_text_best.json")


RS LGB-SVD:   0%|          | 0/15 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

KeyboardInterrupt: 

In [11]:
# # ===== CPU MODE (no CUDA) + sane parallelism =====
# import os, torch, multiprocessing

# # Force CPU everywhere
# device_has_cuda = False
# os.environ["CUDA_VISIBLE_DEVICES"] = ""

# # Use all available vCPUs, control BLAS threads to avoid oversubscription
# CORES = max(2, os.cpu_count() or 4)
# os.environ.update({
#     "OMP_NUM_THREADS": str(CORES),
#     "OPENBLAS_NUM_THREADS": str(CORES),
#     "MKL_NUM_THREADS": str(CORES),
#     "VECLIB_MAXIMUM_THREADS": str(CORES),
#     "NUMEXPR_NUM_THREADS": str(CORES),
# })
# try:
#     torch.set_num_threads(CORES)
#     torch.set_num_interop_threads(max(1, CORES // 2))
# except Exception:
#     pass

# # Split cores between model threads and Optuna parallel trials
# # Option 1 (safer on Colab/CPUs): one trial at a time, max threads per model
# TRIAL_WORKERS = 1
# THREADS_PER_MODEL = CORES

# # Option 2 (throughput): run k trials in parallel, each with fewer threads
# # TRIAL_WORKERS = max(1, CORES // 2)
# # THREADS_PER_MODEL = max(1, CORES // TRIAL_WORKERS)

# print(f"[CPU mode] cores={CORES} | threads/model={THREADS_PER_MODEL} | optuna workers={TRIAL_WORKERS}")


[CPU mode] cores=2 | threads/model=2 | optuna workers=1


In [12]:
# import optuna
# from optuna.pruners import MedianPruner

# optuna.logging.set_verbosity(optuna.logging.WARNING)
# PRUNER = MedianPruner(n_warmup_steps=5)

# # ---- LightGBM on SVD(TF-IDF), CPU ----
# def obj_lgbm_svd(trial):
#     params = {
#         "learning_rate": trial.suggest_float("lr", 0.01, 0.2, log=True),
#         "num_leaves": trial.suggest_int("num_leaves", 16, 256, log=True),
#         "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 10, 200),
#         "feature_fraction": trial.suggest_float("feature_fraction", 0.6, 1.0),
#         "bagging_fraction": trial.suggest_float("bagging_fraction", 0.6, 1.0),
#         "bagging_freq": trial.suggest_int("bagging_freq", 0, 10),
#         "lambda_l1": trial.suggest_float("lambda_l1", 0.0, 5.0),
#         "lambda_l2": trial.suggest_float("lambda_l2", 0.0, 5.0),
#         "objective": "regression",
#         "verbosity": -1,
#         "n_jobs": THREADS_PER_MODEL,   # <— CPU threads
#         # "device_type": "cpu",        # (optional) for native LGBM, wrapper respects CPU by default
#     }
#     oof, _ = cv_lgbm_dense(X_svd, y_log, N_FOLDS, params, early_stop=100)
#     return smape(np.expm1(y_log), np.expm1(oof))

# # ---- XGBoost on SVD(TF-IDF), CPU ----
# def obj_xgb_svd(trial):
#     params = {
#         "tree_method": "hist",          # <— CPU histogram algorithm
#         "predictor": "cpu_predictor",
#         "n_jobs": THREADS_PER_MODEL,    # <— CPU threads
#         "max_depth": trial.suggest_int("max_depth", 3, 10),
#         "min_child_weight": trial.suggest_float("min_child_weight", 1e-2, 20.0, log=True),
#         "subsample": trial.suggest_float("subsample", 0.6, 1.0),
#         "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
#         "eta": trial.suggest_float("eta", 0.01, 0.3, log=True),
#         "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 2.0),
#         "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 2.0),
#     }
#     oof, _ = cv_xgb_dense(X_svd, y_log, N_FOLDS, params, early_stop=200)
#     return smape(np.expm1(y_log), np.expm1(oof))

# # ---- LightGBM on e5 embeddings (subset), CPU ----
# def obj_lgbm_e5(trial):
#     params = {
#         "learning_rate": trial.suggest_float("lr", 0.01, 0.2, log=True),
#         "num_leaves": trial.suggest_int("num_leaves", 16, 256, log=True),
#         "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 10, 200),
#         "feature_fraction": trial.suggest_float("feature_fraction", 0.6, 1.0),
#         "bagging_fraction": trial.suggest_float("bagging_fraction", 0.6, 1.0),
#         "bagging_freq": trial.suggest_int("bagging_freq", 0, 10),
#         "lambda_l1": trial.suggest_float("lambda_l1", 0.0, 5.0),
#         "lambda_l2": trial.suggest_float("lambda_l2", 0.0, 5.0),
#         "objective": "regression",
#         "verbosity": -1,
#         "n_jobs": THREADS_PER_MODEL,   # <— CPU threads
#     }
#     oof_sub, _ = cv_lgbm_dense(Xtr, y_log_sub, N_FOLDS, params, early_stop=100)
#     return smape(np.expm1(y_log_sub), np.expm1(oof_sub))

# # ---- CatBoost on e5 embeddings (subset), CPU ----
# def obj_cat_e5(trial):
#     params = {
#         "depth": trial.suggest_int("depth", 4, 10),
#         "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
#         "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 20.0),
#         "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 5.0),
#         "task_type": "CPU",                 # <— force CPU
#         "thread_count": THREADS_PER_MODEL,  # <— CPU threads
#         "loss_function": "RMSE"
#     }
#     oof_sub, _ = cv_cat_dense(Xtr, y_log_sub, N_FOLDS, params, early_stop=200)
#     return smape(np.expm1(y_log_sub), np.expm1(oof_sub))

# # Trials per study (keep modest on CPU, early stopping cuts work)
# N_TRIALS_LGB_SVD = 20
# N_TRIALS_XGB_SVD = 20
# N_TRIALS_LGB_E5  = 15
# N_TRIALS_CAT_E5  = 15

# # Create studies
# study_lgb_svd = optuna.create_study(direction="minimize", pruner=PRUNER)
# study_xgb_svd = optuna.create_study(direction="minimize", pruner=PRUNER)
# study_lgb_e5  = optuna.create_study(direction="minimize", pruner=PRUNER)
# study_cat_e5  = optuna.create_study(direction="minimize", pruner=PRUNER)

# # Run studies with CPU parallelism
# study_lgb_svd.optimize(obj_lgbm_svd, n_trials=N_TRIALS_LGB_SVD, n_jobs=TRIAL_WORKERS, show_progress_bar=True)
# study_xgb_svd.optimize(obj_xgb_svd, n_trials=N_TRIALS_XGB_SVD, n_jobs=TRIAL_WORKERS, show_progress_bar=True)
# study_lgb_e5.optimize(obj_lgbm_e5,   n_trials=N_TRIALS_LGB_E5,  n_jobs=TRIAL_WORKERS, show_progress_bar=True)
# study_cat_e5.optimize(obj_cat_e5,    n_trials=N_TRIALS_CAT_E5,  n_jobs=TRIAL_WORKERS, show_progress_bar=True)

# best_lgb_svd = study_lgb_svd.best_params
# best_xgb_svd = study_xgb_svd.best_params
# best_lgb_e5  = study_lgb_e5.best_params
# best_cat_e5  = study_cat_e5.best_params

# # Ensure the best params still include CPU threading and predictors
# best_lgb_svd.update({"n_jobs": THREADS_PER_MODEL})
# best_xgb_svd.update({"n_jobs": THREADS_PER_MODEL, "tree_method": "hist", "predictor": "cpu_predictor"})
# best_lgb_e5.update({"n_jobs": THREADS_PER_MODEL})
# best_cat_e5.update({"task_type": "CPU", "thread_count": THREADS_PER_MODEL})

# print("BEST LGB-SVD:", best_lgb_svd)
# print("BEST XGB-SVD:", best_xgb_svd)
# print("BEST LGB-e5 :", best_lgb_e5)
# print("BEST CAT-e5 :", best_cat_e5)
# json.dump({
#     "lgb_svd": best_lgb_svd,
#     "xgb_svd": best_xgb_svd,
#     "lgb_e5": best_lgb_e5,
#     "cat_e5": best_cat_e5,
# }, open(OUT_DIR/"optuna_text_best.json","w"))


  0%|          | 0/20 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[W 2025-10-13 12:37:52,002] Trial 0 failed with parameters: {'lr': 0.06754351367959248, 'num_leaves': 28, 'min_data_in_leaf': 23, 'feature_fraction': 0.8994558424700747, 'bagging_fraction': 0.6080107812105073, 'bagging_freq': 9, 'lambda_l1': 2.9097271997829144, 'lambda_l2': 3.836637714555185} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/optuna/study/_optimize.py", line 201, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipython-input-3484333363.py", line 23, in obj_lgbm_svd
    oof, _ = cv_lgbm_dense(X_svd, y_log, N_FOLDS, params, early_stop=100)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipython-input-1050488292.py", line 12, in cv_lgbm_dense
    m.fit(
  File "/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py", line 1398, in fit
    super().fit(
  File "/usr/local/lib/python3.12/dist-packages/l

KeyboardInterrupt: 

In [ ]:
# LGBM on SVD
oof_lgb_svd, models_lgb_svd = cv_lgbm_dense(X_svd, y_log, N_FOLDS, best_lgb_svd, early_stop=200)
test_lgb_svd = np.mean([m.predict(X_svd_te) for m in models_lgb_svd], axis=0)

# XGB on SVD
oof_xgb_svd, models_xgb_svd = cv_xgb_dense(X_svd, y_log, N_FOLDS, best_xgb_svd, early_stop=400)
test_xgb_svd = np.mean([m.predict(X_svd_te) for m in models_xgb_svd], axis=0)

# LGBM on e5
oof_lgb_e5_sub, models_lgb_e5 = cv_lgbm_dense(Xtr, y_log_sub, N_FOLDS, best_lgb_e5, early_stop=200)
test_lgb_e5 = np.mean([m.predict(Xte) for m in models_lgb_e5], axis=0)

# CatBoost on e5
oof_cat_e5_sub, models_cat_e5 = cv_cat_dense(Xtr, y_log_sub, N_FOLDS, best_cat_e5, early_stop=400)
test_cat_e5 = np.mean([m.predict(Xte) for m in models_cat_e5], axis=0)

# --- PATCH: persist test logs + inflate e5 OOFs for stacking ---

# 1) Save test LOG predictions (the stacking cell loads these)
np.save(OUT_DIR/"test_lgb_svd_log.npy", test_lgb_svd)
np.save(OUT_DIR/"test_xgb_svd_log.npy", test_xgb_svd)
np.save(OUT_DIR/"test_lgb_e5_log.npy",  test_lgb_e5)
np.save(OUT_DIR/"test_cat_e5_log.npy",  test_cat_e5)

# 2) Inflate subset OOF logs from e5 models to full-length arrays
# (cv_* returns LOG-space oof arrays for the subset; fill the rest with NaN)
oof_lgb_e5 = np.full(len(train), np.nan, float); oof_lgb_e5[tr_idx] = oof_lgb_e5_sub
oof_cat_e5 = np.full(len(train), np.nan, float); oof_cat_e5[tr_idx] = oof_cat_e5_sub

# 3) (optional) quick checks so you know files exist before stacking
for fname in ["test_lgb_svd_log.npy","test_xgb_svd_log.npy","test_lgb_e5_log.npy","test_cat_e5_log.npy"]:
    p = OUT_DIR/fname
    print(f"[check] exists: {p} ->", p.exists())



In [ ]:
# Build meta features from LOG OOFs; keep rows where all base OOFs exist
feat_list_log = [
    np.log1p(oof_tfidf_price),             # tfidf ridge (log)
    np.log1p(oof_knn_full),                # e5 kNN (log)
    oof_lgb_svd,                           # already log
    oof_xgb_svd,                           # already log
    oof_lgb_e5,                            # already log (has NaNs)
    oof_cat_e5                             # already log (has NaNs)
]
F = np.vstack(feat_list_log).T
mask = np.all(np.isfinite(F), axis=1)

X_meta = F[mask]
y_meta = y_log[mask]

meta = Ridge(alpha=1.0, random_state=SEED).fit(X_meta, y_meta)

# Build test meta features in the SAME order
test_feat_list_log = [
    np.log1p(test_tfidf_price),
    np.log1p(test_knn_price),
    np.load(OUT_DIR/"test_lgb_svd_log.npy"),
    np.load(OUT_DIR/"test_xgb_svd_log.npy"),
    np.load(OUT_DIR/"test_lgb_e5_log.npy"),
    np.load(OUT_DIR/"test_cat_e5_log.npy"),
]
T = np.vstack(test_feat_list_log).T

# Evaluate meta OOF SMAPE for transparency
meta_oof_log = np.full(len(train), np.nan, float)
meta_oof_log[mask] = meta.predict(X_meta)
meta_oof_price = np.expm1(meta_oof_log)
print("STACK OOF SMAPE:", f"{smape(y_full[mask], meta_oof_price[mask]):.3f}%")

# Final test prediction
test_price_stack = np.expm1(meta.predict(T))


In [ ]:
sub_path = OUT_DIR/"submission_text_stack.csv"
pd.DataFrame({"sample_id": test["sample_id"], "price": test_price_stack}).to_csv(sub_path, index=False)

with open(OUT_DIR/"stack_summary.json","w") as f:
    json.dump({
        "oof_smape": {
            "tfidf_ridge": float(smape(y_full, oof_tfidf_price)),
            "e5_knn": float(smape(y_full, oof_knn_full)),
            "lgb_svd": float(smape(y_full, np.expm1(oof_lgb_svd))),
            "xgb_svd": float(smape(y_full, np.expm1(oof_xgb_svd))),
            "lgb_e5(sub)": float(smape(y_full[np.isfinite(oof_lgb_e5)], np.expm1(oof_lgb_e5[np.isfinite(oof_lgb_e5)]))),
            "cat_e5(sub)": float(smape(y_full[np.isfinite(oof_cat_e5)], np.expm1(oof_cat_e5[np.isfinite(oof_cat_e5)]))),
            "stack(masked)": float(smape(y_full[np.isfinite(meta_oof_price)], meta_oof_price[np.isfinite(meta_oof_price)])),
        }
    }, f, indent=2)

print("Saved:", sub_path)
